In [1]:
import subprocess

def check_and_install_cuda():
    try:
        # Check if nvcc is installed
        subprocess.run(['nvcc', '--version'], check=True, capture_output=True)
        print("nvcc is already installed.")
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("nvcc not found. Installing NVIDIA CUDA toolkit...")
        # Install NVIDIA CUDA Toolkit
        subprocess.run(['apt-get', 'update'], check=True)
        subprocess.run(['apt-get', 'install', '-y', 'nvidia-cuda-toolkit'], check=True)
        print("NVIDIA CUDA toolkit installed.")

check_and_install_cuda()


nvcc is already installed.


In [ ]:
%%writefile main.cpp
<cuda/cmath>

// Kernel definition
__global__ void vecAdd(float* A, float* B, float* C, int vectorLength)
{
  // calculate which element this thread is responsible for computing
  int workIndex = threadIdx.x + blockDim.x * blockIdx.x

  if(workIndex < vectorLength)
  {
    // Perform computation
    C[workIndex] = A[workIndex] + B[workIndex]
  }

}

void unifiedMemExample(int vectorLength)
{
  // Pointers to memory vectors
  float* A = nullptr;
  float* B = nullptr;
  float* C = nullptr;
  float* comparisonResult = (float*)malloc(vectorLength*sizeof(float));

  // Use unified memory to allocate buffers
  cudaMallocManaged(&A, vectorLength*sizeof(float));
  cudaMallocManaged(&B, vectorLength*sizeof(float));
  cudaMallocManaged(&C, vectorLength*sizeof(float));

  // Initialize vectors on the host
  initArray(A, vectorLength);
  initArray(B, vectorLength);

  // Launch
}

int main()
{
  // Kernel invocation
  // Launches a single thread block containing 256 threads,
  // each thread will execute the exact same kernel code.
  int threads = 256;
  int blocks = cuda::ceil_div(vectorLength, threads);
  vecAdd<<<blocks, threads>>>(devA, devB, devC, vectorLength);
  // A, B, and C are vectors of 1024 elements

  // When using 2 or 3-dimensional grids or thread blocks,
  // the CUDA type dim3 is used as the grid and thread block dimension params.
  dim3 grid(16, 16);
  dim3 block(8, 8);
  MatAdd<<<grid, block>>>(A, B, C);
}